# Cloud Workload Forecasting — Exploratory Data Analysis & Validation

This notebook provides exploratory analysis, workload profiling, and temporal validation for the Azure VM Workload Dataset (release `data-v1`).

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from cwfc.data import load_raw_data, build_aggregate_targets, get_train_holdout_split
from cwfc.models import MeanRevertingForecaster, PersistenceForecaster, DirectMultiHorizonForecaster
from cwfc.validation import evaluate_model_rolling_cv
from cwfc.metrics import mae, rmse, smape, normalized_mae

sns.set_theme(style="whitegrid")

## 1. Load Telemetry and Metadata

In [2]:
df_cpu, df_meta = load_raw_data(data_dir="challenge_data")
print(f"CPU Readings Shape: {df_cpu.shape}")
print(f"VM Metadata Shape: {df_meta.shape}")
print(f"Distinct VMs: {df_cpu['vm_id'].nunique():,}")
print(f"Distinct Timestamps: {df_cpu['timestamp'].nunique()}")

## 2. Construct Aggregate Target Time Series
- `active_vm_count(t)`: distinct VMs active at timestamp t.
- `aggregate_cpu_utilization(t)`: mean of `avg_cpu` readings at timestamp t.

In [3]:
df_agg = build_aggregate_targets(df_cpu, df_meta=df_meta)
train_df, holdout_df = get_train_holdout_split(df_agg, train_size=110, holdout_size=24)
print(f"Train observations: {len(train_df)} (~9.17 hours)")
print(f"Holdout observations: {len(holdout_df)} (2.0 hours)")
df_agg.describe().T

## 3. Workload Profiling by Category and Sizing

In [4]:
df_merged = df_cpu.merge(df_meta[["vm_id", "vm_category", "virtual_core_bucket"]], on="vm_id")
cat_summary = df_merged.groupby("vm_category")["avg_cpu"].agg(["count", "mean", "std", lambda x: np.percentile(x, 95)])
cat_summary.columns = ["Readings", "Mean CPU (%)", "Std Dev", "P95 CPU (%)"]
cat_summary

## 4. Expanding-Window Cross-Validation (Horizon = 24)

In [5]:
forecaster = MeanRevertingForecaster(phi=0.90)
cv_res = evaluate_model_rolling_cv(
    forecaster=forecaster,
    train_df=train_df,
    target_col="aggregate_cpu_utilization",
    horizon=24,
    min_train_size=42,
    step=11,
    baseline_forecaster=PersistenceForecaster()
)
print(f"CV Mean MAE: {cv_res['mean_mae']:.4f} +/- {cv_res['std_mae']:.4f}")
print(f"CV Mean RMSE: {cv_res['mean_rmse']:.4f} +/- {cv_res['std_rmse']:.4f}")
print(f"CV Normalized MAE vs Persistence: {cv_res.get('mean_normalized_mae', 1.0):.4f}")